# Ingestion — BGE-M3 on CUDA

The aim of this notebook is to transform legal documents into vector embeddings and upload them to Pinecone.

The embedding model used is **BGE-M3**.

## Setup

1. Attach a GPU-backed kernel.
2. Upload the `Contest_Data/` directory.
3. Upload `Apikey.env`, containing `PINECONE_API_KEY`, to the notebook’s working directory.

## Pinecone index

The `legal-rag` index must have **1,024 dimensions** to match BGE-M3’s embedding output.

## Initial chunking configuration

- Chunk size: **700 words**
- Chunk overlap: **100 words**

These are initial values and can be later evaluated against alternative configurations to determine which provides the best retrieval quality.


## 1.Extract Zip 

In [4]:
import zipfile
with zipfile.ZipFile("Contest_Data.zip") as z:
    z.extractall()

## 2.Imports

In [3]:
import hashlib
import json
import os
from collections import Counter
from pathlib import Path
import re

import torch
import sys
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from tqdm.auto import tqdm  

## 3.Config

In [ ]:
load_dotenv("Apikey.env")

DOCS_DIR = Path("Contest_Data")


INDEX_NAME = "legal-rag"  # Recreated at dimension=1024 on Pinecone 


COUNTRY_MAP = {"italy": "Italy", "estonia": "Estonia", "slovenia": "Slovenia"}


PINECONE_METADATA_LIMIT_BYTES = 40 * 1024


CHUNK_SIZE = 1500
CHUNK_OVERLAP = 150

RESET_INDEX_BEFORE_UPSERT = True

EMBED_MODEL_NAME = "BAAI/bge-m3"

## 4.Startup validation, device check, model + client init

In [ ]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise SystemExit("Missing PINECONE_API_KEY — add it to Apikey.env")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")
if device == "cpu":
    print("  [WARN] No CUDA GPU detected — check this kernel is attached "
          "to a GPU machine. BGE-M3 will be slow on CPU (568M params).")
else:
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

EMBED_MODEL = SentenceTransformer(EMBED_MODEL_NAME, device=device)

pc = Pinecone(api_key=PINECONE_API_KEY)
pine_index = pc.Index(INDEX_NAME)


## 5.Detection of Doc Type

It's purely structural ("Legal Cases" vs "Civil Codes"), law ("Divorce"/"Inheritance") is derived from the folder name for civil codes, and read from each file's own metadata (passthrough, no alias resolution) for cases.

In [ ]:
def detect_doc_type(folder_name: str):
    name = folder_name.lower()
    if "case" in name:
        return "Legal Cases"
    if "divorce" in name or "inheritance" in name:
        return "Civil Codes"
    return None


def normalize_law(raw_law: str, doc_type: str, folder_name: str):
    if doc_type == "Civil Codes":
        name = folder_name.lower()
        if "divorce" in name:
            return "Divorce"
        if "inheritance" in name:
            return "Inheritance"
        return "Unknown"
    return (raw_law or "").strip() or "Unknown"

## 6.MetaData Prepa
Prepares document metadata for Pinecone and embedding.
First, metadata values are converted into Pinecone-compatible types.
Then, useful non-empty fields not already included in the document header
are combined into a readable metadata line, such as CASE_ID, cost,
duration, marital regime, and disputed issues.

In [1]:
def flatten_metadata(raw):
    # None -> "" (Pinecone rejects null metadata values); bool/int/float
    # kept native (so numeric filters like $gt/$lt stay possible later);
    # list -> list of strings (Pinecone arrays must be homogeneous
    # strings); anything else -> str(value).
    flat = {}
    for k, val in raw.items():
        if val is None:
            flat[k] = ""
        elif isinstance(val, bool):
            flat[k] = val
        elif isinstance(val, (int, float)):
            flat[k] = val
        elif isinstance(val, list):
            flat[k] = [str(v) for v in val]
        else:
            flat[k] = str(val)
    return flat


_METADATA_LINE_SKIP_KEYS = {
    "country", "doc_type", "law", "civil_codes_used", "source",
    "text", "chunk_index", "n_chunks",
}
_METADATA_LINE_PLACEHOLDER_VALUES = {"", "no data", "not specified", "n/a", "unknown"}

## 7.Chunking + Embedding-Text Construction 
Prepares document metadata for Pinecone by normalizing unsupported values,
removing technical, duplicate, or empty fields, and combining the remaining
useful information into a readable line included in the embedding text.

In [ ]:
def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
):
    token_ids = EMBED_MODEL.tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    if len(token_ids) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    step = chunk_size - overlap

    while start < len(token_ids):
        end = min(start + chunk_size, len(token_ids))

        chunk = EMBED_MODEL.tokenizer.decode(
            token_ids[start:end],
            skip_special_tokens=True,
        ).strip()

        chunks.append(chunk)

        if end == len(token_ids):
            break

        start += step

    return chunks


def _as_header_str(value) -> str:
    if isinstance(value, list):
        return ", ".join(value)
    return str(value) if value else ""


def build_embedding_text(chunk: str, meta: dict) -> str:
    # Prepend a compact header + the extra-metadata line to the chunk
    # before embedding, e.g.:
    #   [Estonia | Legal Cases | Divorce]
    #   CASE_ID: ...; type: judicial; marital_regime: community of property
    #   <chunk text>
    country  = _as_header_str(meta.get("country", ""))
    doc_type = _as_header_str(meta.get("doc_type", ""))
    law      = _as_header_str(meta.get("law", ""))
    refs     = _as_header_str(meta.get("civil_codes_used", ""))
    meta_line = build_metadata_line(meta)

    header_parts = [p for p in [country, doc_type, law, refs] if p]
    header = "[" + " | ".join(header_parts) + "]" if header_parts else ""

    parts = [p for p in [header, meta_line, chunk] if p]
    return "\n".join(parts)


def make_chunk_id(source: str, chunk_index: int) -> str:
    # Creates a stable ID from the source path and chunk position.
    # Pinecone requires a unique ID for every vector, and this distinguishes
    # multiple chunks from the same document.
    raw = f"{source}::{chunk_index}"
    return hashlib.md5(raw.encode()).hexdigest()

## 8. `load_documents` — parse + validate (no Pinecone, no embedding yet)

Splits long documents into overlapping chunks, enriches each chunk with
relevant metadata, and generates a stable ID before creating the embedding.

In [ ]:
def load_documents(docs_dir):
    docs, errors, unknown_law = [], [], []
    tally = Counter()

    all_files = list(docs_dir.rglob("*.json"))

    for f in tqdm(all_files, desc="Loading documents", unit="doc"):
        folder_country = f.parts[-3].lower()
        folder_type    = f.parts[-2]
        country  = COUNTRY_MAP.get(folder_country)
        doc_type = detect_doc_type(folder_type)
        if not country or not doc_type:
            errors.append(f"{f}: unrecognised folder")
            continue

        try:
            doc = json.loads(f.read_text(encoding="utf-8"))
        except Exception as e:
            errors.append(f"{f.name}: {e}")
            continue

        content = (doc.get("content") or "").strip()
        if not content:
            errors.append(f"{f.name}: empty content")
            continue

        meta = flatten_metadata(doc.get("metadata", {}))
        meta.update({"country": country, "doc_type": doc_type, "source": str(f)})

        law = normalize_law(meta.get("law", ""), doc_type, folder_type)
        meta["law"] = law
        if law == "Unknown":
            unknown_law.append(f"{f.name} (raw law = {doc.get('metadata', {}).get('law')!r})")

        docs.append({"content": content, "metadata": meta})
        tally[(country, doc_type, law)] += 1

    print(f"Loaded: {len(docs)} | Errors: {len(errors)}")
    for e in errors:
        print(f"  [WARN] {e}")

    if unknown_law:
        print(f"\n  [WARNING] {len(unknown_law)} file(s) with unrecognised 'law' "
              f"(ingested as law='Unknown'; Divorce/Inheritance agents will NOT "
              f"find them until fixed):")
        for u in unknown_law:
            print(f"    - {u}")

    print("\n  Summary by (country, type, law):")
    for (c, t, l), n in sorted(tally.items()):
        print(f"    {c:8} | {t:22} | {l:12} : {n} docs")

    return docs

## 9. RUN: load_documents

**Checkpoint.** Read the report before running the next section — check `Errors`, the `Unknown` law list, and whether the per-country/type/law counts look right, before spending GPU time and writing anything to Pinecone.

In [2]:
docs = load_documents(DOCS_DIR)

NameError: name 'load_documents' is not defined

## 10. `build_chunk_records`, metadata size guard, `upsert_docs`

In [ ]:
def build_chunk_records(docs):
    # One entry per source document, each holding its list of chunk
    # records.
    per_doc = []
    for d in docs:
        content = d["content"]
        chunks = chunk_text(content)
        source = d["metadata"].get("source", "")
        chunk_records = []
        for ci, chunk in enumerate(chunks):
            meta = dict(d["metadata"])
            meta["text"] = chunk
            meta["chunk_index"] = ci
            meta["n_chunks"] = len(chunks)
            chunk_records.append({
                "id": make_chunk_id(source, ci),
                "embed_text": build_embedding_text(chunk, meta),
                "metadata": meta,
            })
        per_doc.append({"source": source, "chunks": chunk_records})
    return per_doc


def _fit_metadata(meta: dict) -> dict:
    # Pinecone caps metadata at 40KB/vector. If exceeded, truncate only
    # the "text" field (not the other metadata) to fit.
    size = len(json.dumps(meta).encode("utf-8"))
    if size <= PINECONE_METADATA_LIMIT_BYTES:
        return meta
    meta = dict(meta)
    text = meta.get("text", "")
    overhead = size - len(text.encode("utf-8"))
    max_bytes = max(PINECONE_METADATA_LIMIT_BYTES - overhead - 100, 0)
    meta["text"] = text.encode("utf-8")[:max_bytes].decode("utf-8", errors="ignore")
    return meta


def upsert_docs(docs, batch_size=100):
    # This notebook assumes you always run against an empty or
    # freshly recreated index, not an incremental update over one that
    # already has data from a previous run.
    per_doc = build_chunk_records(docs)
    total_chunks = sum(len(d["chunks"]) for d in per_doc)
    print(f"\n{len(docs)} docs -> {total_chunks} chunks")

    records = [chunk for d in per_doc for chunk in d["chunks"]]

    n_batches = (len(records) + batch_size - 1) // batch_size
    for start in tqdm(
        range(0, len(records), batch_size),
        desc="Embedding + upserting to Pinecone",
        unit="batch",
        total=n_batches,
    ):
        batch = records[start:start + batch_size]
        # normalize_embeddings=True: BGE-M3's own usage docs recommend
        # normalised dense vectors for cosine-similarity retrieval. If
        # your Pinecone index metric isn't cosine, drop this.
        embeds = EMBED_MODEL.encode(
            [r["embed_text"] for r in batch],
            show_progress_bar=False,
            normalize_embeddings=True,
        )
        vectors = []
        for i, r in enumerate(batch):
            meta = _fit_metadata(dict(r["metadata"]))
            vectors.append({
                "id": r["id"],
                "values": embeds[i].tolist(),
                "metadata": meta,
            })
        pine_index.upsert(vectors=vectors)
    print("Done.")

## 11. RUN: upsert_docs

Only run this after checking the report from section 9. This is the cell that writes to Pinecone (index: see `INDEX_NAME` above — confirm it's the 1024-dim one, and empty/freshly recreated).

In [ ]:
upsert_docs(docs)